In [1]:
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from rapidfuzz import fuzz
from itertools import combinations
from shapely.geometry import MultiPoint


# NWT Databases

## OROGO

In [2]:
# OROGO database
OROGO = pd.read_csv('source_data/wells/NWT/orogo-well-status-updated-2026-04-10.csv').drop(columns=['NAD 27 Lat','NAD 27 Long','Unnamed: 16','Unnamed: 17']).dropna(how='all')
OROGO = gpd.GeoDataFrame(data=OROGO,geometry=gpd.points_from_xy(x=OROGO['NAD_83_LongDD'],y=OROGO['NAD_83_LatDD']),crs='NAD1983')
OROGO['Name'] = OROGO['Well Name'].str.upper()
OROGO = OROGO.set_index('Name',drop=False)
OROGO = OROGO.drop(columns='geometry')
OROGO[['y','x']] = OROGO[['NAD_83_LatDD', 'NAD_83_LongDD']]
OROGO['DS_UID'] = [f'OROGO-{i}' for i in range(0,len(OROGO))]
print(len(OROGO))
print(OROGO.columns)

685
Index(['Well ID', 'Well Name', 'Last Operator', 'Current or Last Owner',
       'Well Status', 'Classification', 'First SPUD year',
       'Latest SPUD or Start Date', 'Latest Rig Release or End Date',
       'Land Title', 'Region', 'NAD_83_LatDD', 'NAD_83_LongDD', 'UWI', 'Name',
       'y', 'x', 'DS_UID'],
      dtype='str')


## NTGS Open Report 2009-03

In [3]:
NTGS_2009_03 = [r'source_data\wells\NWT\2009-03_Well_Sites\Well_Sites.shp',r'source_data\wells\NWT\2009-03_Well_Sites\Well_Sites_Peripheral.shp']
NTGS_2009_03 = pd.concat([gpd.read_file(fn).set_index('UWI') for fn in NTGS_2009_03]).to_crs('NAD1983')
NTGS_2009_03['Name'] = NTGS_2009_03['WELL_NAME'].str.upper()
NTGS_2009_03['NAD_83_LatDD'] = NTGS_2009_03.geometry.y
NTGS_2009_03['NAD_83_LongDD'] = NTGS_2009_03.geometry.x
NTGS_2009_03 = NTGS_2009_03.reset_index().set_index('Name',drop=False)
NTGS_2009_03 = NTGS_2009_03.drop(columns='geometry')
NTGS_2009_03[['y','x']] = NTGS_2009_03[['NAD_83_LatDD', 'NAD_83_LongDD']]
NTGS_2009_03['DS_UID'] = [f'NTGS_2009_03-{i}' for i in range(0,len(NTGS_2009_03))]
print(len(NTGS_2009_03))
print(NTGS_2009_03.columns)

549
Index(['UWI', 'WID', 'WELL_NAME', 'OPERATOR', 'STATUS', 'CLASSIFICA',
       'SPUD_DATE', 'RIG_RELEAS', 'LAT', 'LONG', 'DATUM', 'NORTHING',
       'EASTING', 'ZONE', 'SOURCE', 'REFERENCE', 'Name', 'NAD_83_LatDD',
       'NAD_83_LongDD', 'y', 'x', 'DS_UID'],
      dtype='str')


## NTGS Open Report 2019-015

In [ ]:
# Report 2019-015
NTGS_2019_015_g = gpd.read_file(r'source_data\wells\NWT\2019-015_Shapefiles\Gas_Resource.shp').set_index('Well_Names',drop=False).rename(columns={'Recoverabl':'Recoverabl_gas'})
NTGS_2019_015_o = gpd.read_file(r'source_data\wells\NWT\2019-015_Shapefiles\Oil_Resource.shp').set_index('Well_Names',drop=False).rename(columns={'Recoverabl':'Recoverabl_oil'})

NTGS_2019_015 = pd.concat([
    NTGS_2019_015_g,
    NTGS_2019_015_o.loc[~NTGS_2019_015_o.index.isin(NTGS_2019_015_g.index),NTGS_2019_015_o.columns[NTGS_2019_015_o.columns!='Recoverabl_oil']]
]).to_crs('NAD1983')
NTGS_2019_015 = NTGS_2019_015.join(NTGS_2019_015_o[['Recoverabl_oil']],how='left')
NTGS_2019_015['NAD_83_LatDD'] = NTGS_2019_015.geometry.y
NTGS_2019_015['NAD_83_LongDD'] = NTGS_2019_015.geometry.x
NTGS_2019_015 = NTGS_2019_015.drop(columns='geometry')
NTGS_2019_015[['y','x']] = NTGS_2019_015[['NAD_83_LatDD', 'NAD_83_LongDD']]
NTGS_2019_015['Name'] = NTGS_2019_015['Well_Names'].str.upper()
NTGS_2019_015 = NTGS_2019_015.set_index('Name',drop=False)

NTGS_2019_015 = NTGS_2019_015.sort_index()
NTGS_2019_015 = NTGS_2019_015.loc[~NTGS_2019_015[['UWI']].duplicated(keep='first')]
NTGS_2019_015['DS_UID'] = [f'NTGS_2019_015-{i}' for i in range(0,len(NTGS_2019_015))]

print(len(NTGS_2019_015))
print(NTGS_2019_015.columns)

167
Index(['NEB_Study', 'NEB_Field', 'WID', 'UWI', 'Well_Names', 'Discovery',
       'Field_Code', 'Pool_Codes', 'Pool_Seq', 'Fluid_Type', 'Initial_Ma',
       'Recoverabl_gas', 'Bot_Hole_L', 'Bot_Hole_1', 'Recoverabl_oil',
       'NAD_83_LatDD', 'NAD_83_LongDD', 'y', 'x', 'Name', 'DS_UID'],
      dtype='str')


# Yukon Data

In [5]:
GeoYukon = gpd.read_file(r'source_data\wells\yukon\Oil_and_Gas_Wells_50k.shp').to_crs('NAD1983').rename(columns={'WELL_UWI':'UWI'})
GeoYukon['Name'] = GeoYukon['WELL_NAME'].str.upper()
GeoYukon = GeoYukon.set_index('Name',drop=False)
GeoYukon['NAD_83_LatDD'] = GeoYukon.geometry.y
GeoYukon['NAD_83_LongDD'] = GeoYukon.geometry.x
GeoYukon = GeoYukon.drop(columns='geometry')
GeoYukon[['y','x']] = GeoYukon[['NAD_83_LatDD', 'NAD_83_LongDD']]
GeoYukon['DS_UID'] = [f'GeoYukon-{i}' for i in range(0,len(GeoYukon))]
print(len(GeoYukon))
print(GeoYukon.columns)

76
Index(['WELL_NAME', 'WELL_LABEL', 'LIC_NUM', 'UWI', 'LICENSEE', 'CLASS',
       'TYPE', 'STATUS', 'DATE', 'OPERATOR', 'PROD_DATE', 'ABANDON',
       'LOCATION', 'BASIN_NAME', 'LAT_DD', 'LONG_DD', 'Name', 'NAD_83_LatDD',
       'NAD_83_LongDD', 'y', 'x', 'DS_UID'],
      dtype='str')


# Federal Data

## Basin

In [6]:
Basin = pd.read_csv('source_data/wells/Basin/BASIN_well_coords.txt',delimiter='\t',skiprows=3).drop(columns=['Latitude (NAD27)','Longitude (NAD27)','Northing (NAD27)','Easting (NAD27)'])
Basin2 = pd.read_csv('source_data/wells/Basin/q1787332704.txt',delimiter='\t',skiprows=3).rename(columns={'Unique Well Identifier':'UWI'})
Basin2['UWI'] = Basin2['UWI'].str.replace(' ','')
Basin = pd.merge(Basin,Basin2[['Well Name','GSC #','Original Spud Year','Operator','Status','UWI']],on='Well Name',how='left')#.set_index('UWI')
Basin['Name'] = Basin['Well Name'].str.upper()
Basin = Basin.set_index('Name',drop=False)
Basin = Basin.sort_values(by='Original Spud Year')
Basin = Basin.loc[~Basin[['Name','UWI']].duplicated(keep='first')].copy()
Basin = Basin.loc[Basin['Latitude (NAD83)']>60]
Basin[['y','x']] = Basin[['Latitude (NAD83)', 'Longitude (NAD83)']]
Basin['DS_UID'] = [f'Basin-{i}' for i in range(0,len(Basin))]
print(len(Basin))
print(Basin.columns)

179
Index(['Well Name', 'Basin', 'Subbasin', 'Latitude (NAD83)',
       'Longitude (NAD83)', 'Northing (NAD83)', 'Easting (NAD83)',
       'Zone (UTM)', 'GSC #', 'Original Spud Year', 'Operator', 'Status',
       'UWI', 'Name', 'y', 'x', 'DS_UID'],
      dtype='str')


## CER

* Missing UWI

In [7]:
CER = pd.read_csv(r'source_data\wells\CER\Frontier_Wells_Inuvik_Region.csv')
CER['Name'] = CER['WELL_NAME'].str.upper()
CER['UWI'] = None
CER = CER.set_index('Name',drop=False)
CER[['y','x']] = CER[['NAD_83_LAT', 'NAD_83_LON']]
CER['DS_UID'] = [f'CER-{i}' for i in range(0,len(CER))]
print(len(CER))
print(CER.columns)

367
Index(['WELL_ID', 'WELL_NAME', 'OPERATOR', 'STATUS', 'LAND_TITLE', 'REGION',
       'NAD_83_LAT', 'NAD_83_LON', 'Name', 'UWI', 'y', 'x', 'DS_UID'],
      dtype='str')


## GSC Open Files 

* some missing key identifiers, so pre-synthesis needed for better record matching

## GSC Open File 6959

In [8]:
GSC_6959 = pd.read_csv(r'source_data\wells\GSC\GSC_OF_6959.csv')#@.set_index('UWI')
GSC_6959['Name'] = GSC_6959['Well Short Name'].str.upper()
GSC_6959=GSC_6959.set_index('Name',drop=False)
GSC_6959[['y','x']] = GSC_6959[['SURF_LAT', 'SURF_LONG']]
GSC_6959['DS_UID'] = [f'GSC_6959-{i}' for i in range(0,len(GSC_6959))]
print(len(GSC_6959))
print(GSC_6959.columns)

265
Index(['UWI', 'Well Short Name', 'EASTING', 'NORTHING', 'MAP_DATUM',
       'SURF_LAT', 'SURF_LONG', 'UTMZONE', 'Well Status', 'KB (m)', 'GL (m)',
       'Final Interpretation, Ice-bearing permafrost base, Base of fully frozen (mKB)',
       'Final Interpretation Ice-bearing permafrost base Base of fully frozen (mGL/SF)',
       'Final Interpretation Ice-bearing permafrost base Quality', 'Formation',
       'Final Interpretation Ice-bearing permafrost base Base of partially frozen (mKB)',
       'Final Interpretation Ice-bearing permafrost base Base of partially frozen (mGL/SF)',
       'Final Interpretation Ice-bearing permafrost base Quality.1',
       'Final Interpretation Ice-bearing permafrost base Transition zone thickness (m)',
       'Final Interpretation Ice-bearing permafrost base Formation/ Sequence ',
       'Final Interpretation Ice-bearing permafrost base PF present',
       'Final Interpretation Ice-bearing permafrost base Data used for final pick*',
       'Base of 

## GSC Open File 4828 & 327948

In [9]:
# Missing UWI
GSC_4828 = pd.read_csv(r'source_data\wells\GSC\GSC_OF_4828.csv')
for c in ['Latitude', 'Longitude']:
    DMS = pd.DataFrame(GSC_4828[c].replace(r'[^0-9.]',' ',regex=True).str.split().to_list(),columns=['D','M','S'],index=GSC_4828['Well Name'])
    GSC_4828[c] = (DMS['D'].astype('float')+DMS['M'].astype('float')/60+DMS['S'].astype('float')/3600).values
GSC_4828['Longitude'] *= -1
GSC_4828['Name'] = GSC_4828['Well Name'].str.upper()
GSC_4828 = GSC_4828.set_index('Name',drop=False)
GSC_4828[['y','x']] = GSC_4828[['Latitude', 'Longitude']]

# # Missing coordinates but contains UWI
GSC_327948 = pd.read_csv(r'source_data\wells\GSC\GSC_OF_327948.csv')
GSC_327948['Name'] = GSC_327948['Well name'].str.upper()
GSC_327948 = GSC_327948.set_index('Name',drop=False)
GSC_327948 = GSC_327948.loc[GSC_327948['UWI'].duplicated(keep='first')].copy()
GSC_327948[['x','y']] = None,None
# Get from GSC_6959 & 327948
GSC_4828=GSC_4828.join(GSC_6959[['UWI']],how='left')
for k,v in GSC_4828.loc[GSC_4828['UWI'].isna(),'Name'].isin(GSC_327948['Name']).items():
    if v:
        GSC_4828.loc[GSC_4828.index==k,'UWI'] = GSC_327948.loc[GSC_327948.index==k,'UWI']
GSC_4828['DS_UID'] = [f'GSC_4828-{i}' for i in range(0,len(GSC_4828))]
GSC_327948['DS_UID'] = [f'GSC_327948-{i}' for i in range(0,len(GSC_327948))]
print(len(GSC_4828))
print(GSC_4828.columns)

print(len(GSC_327948))
print(GSC_327948.columns)


263
Index(['Well No.', 'Company', 'Well Name', 'Latitude', 'Longitude', 'KBm',
       'GLm', 'TD(ft)', 'TD(m)', 'TVD', 'Name', 'y', 'x', 'UWI', 'DS_UID'],
      dtype='str')
40
Index(['UWI', 'Well name', 'Top of overpressure zone mSl',
       'Top of overpressure zone mGl', 'Top of overpressure zone quality',
       'Sequence', 'Comments', 'Name', 'x', 'y', 'DS_UID'],
      dtype='str')


# Consultant Reports

## Arktis report for ILA

In [10]:
ILA = pd.read_csv(r'source_data\wells\ISR\ISR_well_report_B1.csv')
ILA = ILA.loc[~ILA['Latitude (NAD83) -Well Post'].isna()].copy()
for c in ['Latitude (NAD83) -Well Post','Longitude (NAD83) -Well Post']:
    DMS = pd.DataFrame(ILA[c].replace(r'[^0-9.]',' ',regex=True).str.split().to_list(),columns=['D','M','S'],index=ILA['Well Name'])
    ILA[c] = (DMS['D'].astype('float')+DMS['M'].astype('float')/60+DMS['S'].astype('float')/3600).values
ILA['Longitude (NAD83) -Well Post'] *= -1
ILA['Name'] = ILA['Well Name'].str.upper()
ILA = ILA.set_index('Name',drop=False)
ILA[['y','x']] = ILA[['Latitude (NAD83) -Well Post','Longitude (NAD83) -Well Post']]

ILA['DS_UID'] = [f'ILA-{i}' for i in range(0,len(ILA))]
print(len(ILA))
print(ILA.columns)

227
Index(['WID', 'Consortium', 'Current Owner', 'Land Owner', 'Well Name', 'UWI',
       'Class', 'Status', 'Latitude (NAD83) -Well Post',
       'Longitude (NAD83) -Well Post', 'Region', 'Original Spud Date',
       'Original Rig Release Date', 'Depth (m)', 'Notes', 'Name', 'y', 'x',
       'DS_UID'],
      dtype='str')


## LTLC Report

In [11]:
LTLC = pd.read_csv('source_data/wells/misc/LTLC_report.csv')
LTLC = LTLC.set_index('Name',drop=False)
LTLC['UWI'] = None
LTLC[['x','y']] = None,None


LTLC['DS_UID'] = [f'LTLC-{i}' for i in range(0,len(LTLC))]
print(len(LTLC))
print(LTLC.columns)

92
Index(['Name', 'Operator', 'Spud_Date', 'RIG RELEASE', 'DRILLING PLATFORM',
       'WATER DEPTH (M)', 'UWI', 'x', 'y', 'DS_UID'],
      dtype='str')


# Aggregating

In [ ]:
ds = {
'OROGO':OROGO,
'NTGS_2009_03':NTGS_2009_03,
'NTGS_2019_015':NTGS_2019_015,
'GeoYukon':GeoYukon,
'Basin':Basin,
'CER':CER,
'GSC_327948':GSC_327948,
'GSC_4828':GSC_4828,
'GSC_6959':GSC_6959,
'ILA':ILA,
'LTLC':LTLC
}
# Corrections
for key,value in ds.items():
	df = ds[key]
	# A UID for backtracking all records
	df['Data_Source'] = key
	df['UWI'] = df['UWI'].fillna('None').astype('str').str.lstrip()
	# consolidate points, round to ~ 10m
	ds[key][['x','y']] = ds[key][['x','y']].astype('float').round(4)
AllData = pd.concat([d[['Name','UWI','Data_Source','DS_UID','x','y']] for d in ds.values()])
AllPoints = AllData.groupby(['x','y']).agg(set)
AllPoints['replicates'] = AllData.groupby(['x','y']).count()['DS_UID']
AllPoints['OID'] = [i for i in range(0,len(AllPoints))]
AllPoints = AllPoints.reset_index().set_index('OID').sort_values(by='replicates')

# Multi = AllPoints.loc[AllPoints['replicates']>1].copy()

from rapidfuzz import fuzz
from itertools import combinations
from shapely.geometry import multipoint

def filter(X,mask=['None',None,np.nan],fuzzmask=85):
	out = [x for x in X if x not in mask]
	if len(out) == 0:
		return (None)
	elif len(out)==1:
		return(out[0])
	elif fuzzmask is None:
		return(out)
	else:
		fuzzyMin = min([fuzz.partial_ratio(c[0],c[1]) for c in combinations(out,2)])
		if fuzzyMin>=fuzzmask:
			out.sort()
			out = out[0]
		return(out)

def listCheck(X):
	return(type(X) is list)

for c in ['Data_Source','DS_UID']:
	AllPoints[c] = AllPoints[c].apply(list).apply(';'.join)

#Fuzzymatch similar values	
AllPoints['Name'] = AllPoints['Name'].apply(list).apply(lambda x: filter(x,fuzzmask=80))
AllPoints['UWI'] = AllPoints['UWI'].apply(list).apply(lambda x: filter(x,fuzzmask=90))

ix = ((~AllPoints['Name'].apply(listCheck)&(AllPoints['UWI'].apply(listCheck))))
for i,row in AllPoints.loc[ix].iterrows():
	row['UWI'].sort()
	AllPoints.loc[i,'UWI'] = row['UWI'][0]

ix = ((AllPoints['Name'].apply(listCheck)&(~AllPoints['UWI'].apply(listCheck))))
for i,row in AllPoints.loc[ix].iterrows():
	row['Name'].sort()
	AllPoints.loc[i,'Name'] = row['Name'][0]

ix = ((AllPoints['Name'].apply(listCheck)&(AllPoints['UWI'].apply(listCheck))))
for i,row in AllPoints.loc[ix].iterrows():
	row['Name'].sort()
	AllPoints.loc[i,'Name'] = row['Name'][0]
	row['UWI'].sort()
	AllPoints.loc[i,'UWI'] = row['UWI'][0]

# def diff(X)

AllUWI = AllPoints.groupby('UWI').agg(list)#['replicates'].apply(sum)
AllUWI['Name'] = AllUWI['Name'].apply(set).apply(lambda x: filter(x,fuzzmask=80))
AllUWI['Data_Source'] = AllUWI['Data_Source'].apply(';'.join)
AllUWI['DS_UID'] = AllUWI['DS_UID'].apply(';'.join)
AllUWI['replicates'] = AllUWI['replicates'].apply(sum)
AllUWI['Sources'] = AllUWI['Data_Source'].str.split(';').str.len()

columns_by_surce = {
'Company':{
	'OROGO':'Current or Last Owner',
	'NTGS_2009_03':'OPERATOR',
	'NTGS_2019_015':None,
	'GeoYukon':'LICENSEE',
	'Basin':'Operator',
	'CER':'OPERATOR',
	'GSC_327948':None,
	'GSC_4828':'Company',
	'GSC_6959':None,
	'ILA':'Current Owner',
	'LTLC':'Operator'
		   },
}

lx = 0
for col,parameters in columns_by_surce.items():
	AllUWI[col] = [[] for i in range(0,len(AllUWI))]
	for source,param in parameters.items():
		if param is None:
			continue


# for source,parameters in columns_by_surce.items():
# 	for col,param in parameters.items():
		d = ds[source]
		for ix,UID in AllUWI.loc[AllUWI['Data_Source'].str.contains(source),'DS_UID'].str.split(';').items():
			UID = [uid for uid in UID if uid.startswith(source)]
			for uid in UID:
				AllUWI.loc[ix,'Company'].append(d.loc[d['DS_UID']==uid,param].values[0])
	AllUWI[col] = AllUWI[col].apply(lambda x: filter(x,fuzzmask=80))
AllUWI

# for ix,val in AllUWI['DS_UID'].items():
# 	for value in val:
# 		d=value.split('-')[0]
# 		print(d)
# 		for k,v in ds.items():
# 			print(ds[d].columns)
# 			print(ds[d].loc[ds[d]['DS_UID']==value,v])

# geom = []
# for i,row in AllUWI.iterrows():
# 	geom.append(MultiPoint([(x,y) for x,y in zip(row['x'],row['y'])]))
# gdf = gpd.GeoDataFrame(data=AllUWI,geometry=geom,crs='NAD1983')
# gdf.plot(column='Sources',legend=True)
# gdf.loc[gdf['Sources']>=5]#gdf['Sources'].max()]

# AllUWI['DS_UID']#.str.len().max()

,x,y,Name,Data_Source,DS_UID,replicates,Sources,Spud_Date,Company
UWI,,,,,,,,,
300A016010123151,[-123.2531],[60.0011],FORT LIARD A-01,OROGO,OROGO-574,1,1,None,Paramount Resources Ltd.
300A016850134000,"[-134.0112, -134.0086]","[68.6702, 68.6703]",IKHIL A-01,GSC_6959;ILA;GSC_4828;CER,GSC_6959-56;ILA-41;CER-57;GSC_4828-28,4,4,None,"[Gulf Canada Resources Inc., Gulf et al., Cono..."
300A027940087000,[-87.0204],[79.521],MOKKA A-02,Basin,Basin-37,1,1,None,IMP PANARCTIC ET AL
300A036010117300,[-117.5017],[60.0369],CAMERON A-03,OROGO,OROGO-661,1,1,None,Strategic Oil & Gas Ltd.
300A056010117300,[-117.5089],[60.0674],CAMERON A-05,NTGS_2019_015,NTGS_2019_015-26,1,1,None,None
...,...,...,...,...,...,...,...,...,...
307O376520126450,[-126.8554],[65.2833],NORMAN WELLS C-38X,NTGS_2009_03,NTGS_2009_03-288,1,1,None,Imperial Oil Limited
307P376520126450,[-126.8492],[65.2838],NORMAN WELLS C-40X,NTGS_2009_03,NTGS_2009_03-286,1,1,None,Imperial Oil Limited
308A486520126450,[-126.8798],[65.2836],NORMAN WELLS H-29X,NTGS_2009_03,NTGS_2009_03-242,1,1,None,Imperial Oil Limited


In [ ]:
ds = {
'OROGO':OROGO,
'NTGS_2009_03':NTGS_2009_03,
'NTGS_2019_015':NTGS_2019_015,
'GeoYukon':GeoYukon,
'Basin':Basin,
'CER':CER,
'GSC_327948':GSC_327948,
'GSC_4828':GSC_4828,
'GSC_6959':GSC_6959,
'ILA':ILA,
'LTLC':LTLC
}
# Corrections
for key,value in ds.items():
	# A UID for backtracking all records
	if 'UID' not in ds[key].columns:
		ds[key]['UID'] = [f'{key}-{i}' for i in range(0,len(ds[key]))]
	ds[key]['Data_Source'] = key
	ds[key]['UWI'] = ds[key]['UWI'].fillna(None).astype('str').str.lstrip()

	ds[key][['x','y']] = ds[key][['x','y']].astype('float').round(4)
	# Remove duplicated coordinates within same layer
	ds[key] = ds[key].sort_index()
	# print('spatial duplicate: ',key,ds[key].loc[((ds[key][['x','y']].duplicated())&(~ds[key]['x'].isna()))].shape[0])
	# ds[key] = ds[key].loc[((~ds[key][['x','y']].duplicated())&(~ds[key]['x'].isna()))].copy()
	# print(ds[key].loc[((~ds[key][['x','y']].duplicated())&(~ds[key]['x'].isna()))])

	# Ensure north of 60
	ds[key] = ds[key].loc[ds[key]['y']>60].copy()


	# Fix two UWI inconsistencies
	if key == 'NTGS_2009_03':
		ds[key]['UWI'] = ds[key]['UWI'].replace({'300F296830134300':'300F297000134000'})
	elif key == 'GSC_327948':
		ds[key]['UWI'] = ds[key]['UWI'].replace({'302I457030133302':'302P357020134000'})

	# Fix abbreviations in names to better sny duplicates
	rep = {
		'N.W.': 'NORTHWEST',
		'N.E.': 'NORTHEAST',
		'S.W.': 'SOUTHWEST',
		'S.E.': 'SOUTHEAST',
		'N.': 'NORTH',
		'S.': 'NORTH',
		'W.': 'WEST',
		'E.': 'EAST',
		'Y.T.': 'YT',
		'YT.': 'YT',
		'PEEL R ': 'PEEL RIVER ',
		' R. ': ' RIVER ',
		' CK ':' CR ',
	}
		
	for pat_in,pat_out in rep.items():
		ds[key]['Name'] = ds[key]['Name'].str.replace(pat_in,pat_out)

	# Name issues
	rep = {
		'REINDEER F-36':'REINDEER C-36',
		'REINDEER C-36 (F-36)':'REINDEER C-36',
		'PACIFIC ET AL PEEL YT F-37':'PEEL YT F-37',
		'IKHIL A-01 (EAST REINDEER A-01)':'IKHIL A-01',
		'TARSUIT A-25':'TARSIUT A-25',
		'ONIGAT C-38 (EAST REINDEER C-38)':'ONIGAT C-38',
		'UKALERK C-50-2':'UKALERK 2C-50',
		'ATIGI G-04 (EAST REINDEER G-04)':'ATIGI G-04',
		'SHOLOKPAOQAK P-60 (EAST REINDEER P-60)':'SHOLOKPAOQAK P-60',
		'RAMPARTS NO.1(I-55)':'RAMPARTS 1 I-55',
		'RAMPARTS NO. 1 (I-55)':'RAMPARTS 1 I-55',
		'DISCOVERY NO. 1':'DISCOVERY 1',
		'DISCOVERY NO.1':'DISCOVERY 1',
		'BEAVER HOUSE':'BEAVERHOUSE',
		'LABICHE':'LA BICHE',
		'NORTH RAMPARTS I-77':'SOUTH RAMPARTS I-77',
		'NORTH MAIDA CREEK G-56':'SOUTH MAIDA CREEK G-56',
		'KOAKAOK O-22':'KOAKOAK O-22',
		'AMERADA ET AL CROWN YT-A NO 1 N-50':'YT-A 1 N-50',
		'RIVER "YT-A" NO.1 (N-50)':'YT-A 1 N-50',
		'RABBIT LAKE NO.1':'RABBIT LAKE 1 ',
		'RABBIT LAKE NO. 1':'RABBIT LAKE 1',
		'YA-YA':'YA YA',
		'YAYA':'YA YA',
		'BAYYTL':'BAY YT L',
		'LK S':'LAKE NORTH',
		'NORTH TUTTLE YT -05':'SOUTH TUTTLE YT N-05',
		'S TUTTLE ':'SOUTH TUTTLE ',
		'ATERTAK K-31':'ATERTAK L-31',
		'N PARK':'NORTH PARK',
		'N CAM': 'NORTH CAM',
		'LIARD F-25A':'LIARD F-25',
		'E PINE':'EAST PINE',
		'KURK M-15':'KURK M-15',
		'TAGLU D-43 (F-43)':'TAGLU D-43',
		'PARSONS E-02 (F-02)':'PARSONS E-02',
		'NUNA E-40 (D-40)':'NUNA E-40',
		'ATERTAK L-31 (K-31)':'ATERTAK L-31',
		'TUK L-09 (M-09)':'TUK L-09'
		}
	
	for pat_in,pat_out in rep.items():
		ds[key]['Name'] = ds[key]['Name'].str.replace(pat_in,pat_out)
	

	# Remove company name from well name
	drop = [
		'AMOCO PCP B-1 ',
		'SOCONY MOBIL WM ',
		'SKELLY GETTY MOBIL ',
		'SHELL ',
		'CHEVRON SOBC WM ',
		'CHEVRON SOBC GULF ',
		'CHEVRON SOBC IMP ',
		'IOE ',
		'MOBIL GULF ',
		'MCD GCO NORTHUP ',
		'GULF MOBIL ',
		'PACIFIC IMP ET AL ',
		'SOBC WM ',
		'TOLTEC ',
		'CROWN BELL',
		]
	for d in drop:
		ds[key]['Name'] = ds[key]['Name'].str.replace(d,'').str.lstrip()

	# Some records have "extra" names denoted by "; or in ()"
	ds[key]['Name'] = ds[key]['Name'].str.split(';').str[0]
	# ds[key]['Name'] = ds[key]['Name'].str.split(r'\(').str[0].str.rstrip()
	ds[key] = ds[key].set_index('Name',drop=False)
	ds[key]['UWI'] = ds[key]['UWI'].fillna('None')
	

# Fill missing UWI
AllSources = pd.concat([d[['Name','UWI','Data_Source','x','y']] for d in ds.values()]).sort_index()
# AllSources['UWI'] = AllSources['UWI'].fillna('None')
for i,row in AllSources.groupby(AllSources.index).agg(list).iterrows():
	if len(row['Data_Source']) == 1:
		pass
	elif len(row['Data_Source']) > 1:
		UWI_set = set(row['UWI'])
		if 'None' in UWI_set and len(UWI_set)==2:
			UWI = ''.join([f"{u}" for u in UWI_set if u != 'None'])
			# Fill missing UWI by name
			sources = [row['Data_Source'][i] for i,v in enumerate(row['UWI']) if v == 'None']
			for s in sources:
				ds[s].loc[ds[s].index==i,'UWI']=UWI
		elif len(UWI_set)>1:
			# # Some UWI match between datasets except they have 1 or 2 at end instead of 0, this could be a typo, or a record of "events",
			# # but if they have matching names, the i assume they are the "same" spot
			# # even though the coordinates may not match.
			if len(set([uwi[:15] for uwi in row['UWI'] if uwi != 'None'])) == 1:
				UWI = [r for r in row['UWI'] if r != 'None'][0][:15]+'0'
				for s in sources:
					ds[s].loc[ds[s].index==i,'UWI']=UWI
# for i,row in AllSources.loc[]

				
AllSources = pd.concat([d[['Name','UWI','Data_Source','x','y']] for d in ds.values()]).sort_index()
for i,row in AllSources.loc[AllSources['UWI']=='None'].iterrows():
	nuwi = 'NA_UWI_'+re.sub('[^a-zA-Z0-9]','',row['Name'])[:10]
	# print(i,row['Data_Source'],nuwi)
	ds[row['Data_Source']].loc[ds[row['Data_Source']].index==i,'UWI'] = nuwi
	# print(ds[row['Data_Source']].loc[ds[row['Data_Source']].index==i,'UWI'])
	# print()

AllSources = pd.concat([d[['Name','UWI','Data_Source','x','y']] for d in ds.values()]).sort_index()
SourceCounts = AllSources.groupby('UWI').agg(list)['Data_Source'].sort_values()
SingleSource = SourceCounts.loc[SourceCounts.str.len()==1]#.index
print(len(SingleSource))
MultiSource = SourceCounts.loc[SourceCounts.str.len()>1]
print(len(MultiSource))
MultiSource

1141
517


UWI
300C587620111000              [Basin, CER]
300I347630113000              [Basin, CER]
300C737540111300         [Basin, CER, ILA]
300H497700118300         [Basin, CER, ILA]
300K337650113300         [Basin, CER, ILA]
                             ...          
300M056020118150    [OROGO, NTGS_2019_015]
300A636050118000    [OROGO, NTGS_2019_015]
300G636020115450    [OROGO, NTGS_2019_015]
300J626010117150    [OROGO, NTGS_2019_015]
300I196100122300    [OROGO, NTGS_2019_015]
Name: Data_Source, Length: 517, dtype: object

In [418]:
# # set('None') == set('None')
# SingleSource.sort_values().str[0]
# ds['Basin'].loc[ds['Basin']['UWI']=='NA_UWI_NUKIK1']
# for key,value in ds.items():
    
#     print(key,value.columns)
# Duplicates = AllSources.loc[AllSources[['x','y','UWI']].duplicated(keep=False)]
# AllSources.loc[((AllSources['UWI'].duplicated(keep=False))&(~AllSources[['x','y',]].duplicated(keep=False)))]

Points = {}
AllPoints = AllSources.groupby(['x','y']).agg(list)
for i,row in AllPoints.iterrows():
    # Points[i]='x'
    # print(i,row)
    if len(set(row['UWI']))>1:
        print(row)
        


Name           [EAST PINE CREEK YT O-78, EAST PINE CREEK YT O...
UWI                         [300O786700137452, 300O786700137450]
Data_Source                             [NTGS_2009_03, GeoYukon]
Name: (-137.9853, 66.9645), dtype: object
Name                 [BIRCH YT B-34, BIRCH YT B-34]
UWI            [300B346610136450, 300B346610136451]
Data_Source                [GeoYukon, NTGS_2009_03]
Name: (-136.8572, 66.0507), dtype: object
Name               [TARSIUT P-45, WEST TARSIUT P-45]
UWI            [NA_UWI_TARSIUTP45, 300P457000136150]
Data_Source                          [GSC_4828, CER]
Name: (-136.418, 69.9154), dtype: object
Name           [EAST TARSIUT N-44, EAST TARSIUT N-44A]
UWI               [300N447000136000, 300N447000136001]
Data_Source                            [GSC_4828, CER]
Name: (-136.1941, 69.8969), dtype: object
Name           [EAST TARSIUT N-44, EAST TARSIUT N-44A]
UWI               [300N447000136000, 300N447000136001]
Data_Source                            [CER, GSC